# Co2L en Task-IL (Plan de Implementacion)

Notebook guia para implementar Co2L (Contrastive Continual Learning) sobre Seq-CIFAR-10 en este repo.

Objetivo: pasar de LwF basado en logits a Co2L basado en representaciones (SupCon + distillation de features/proyecciones).

## Paso a Paso (Roadmap)

**Nota:** Co2L es **decoupled** (representation ≠ classifier). NO usa CE loss durante Phase 1.

1. Preparar dos vistas por imagen para contraste. ✅
2. Agregar projection head al modelo. ✅
3. Implementar losses Co2L (L_sup_asym + L_IRD, **NO CE**). ✅
4. Implementar entrenamiento representation (Phase 1) con teacher congelado. ✅
5. (TODO) Entrenar classifier linealmente en Phase 2 sobre features congelados.
6. Integrar replay buffer y evaluacion Task-IL.
7. Medir forgetting y guardar checkpoints por tarea.


In [ ]:
# Setup base
from copy import deepcopy
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from dataloaders import SequentialCIFAR10, TwoViewWrapper
from models import CNN, TaskIncrementalClassifier
from train import evaluate_task_incremental
from utils import save_task_incremental_classifier

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 128
NUM_TASKS = 5
CO2L_CKPT_DIR = "checkpoints/co2l_task_il"
os.makedirs(CO2L_CKPT_DIR, exist_ok=True)

seq_cifar = SequentialCIFAR10(data_root="./data", batch_size=BATCH_SIZE, buffer_size=200)
print(f"Device: {device}")

## TODO Global

- [x] Definir wrapper de dataset para devolver 2 vistas por muestra.
- [x] Definir projection head y forward contrastivo.
- [x] Implementar Co2L losses (L_sup_asym + L_IRD, decoupled, NO CE).
- [x] Implementar teacher snapshot por tarea.
- [x] Implementar Phase 1 (representation learning con L_sup_asym + L_IRD).
- [x] Implementar Phase 2 (linear classifier training en representaciones congeladas).
- [ ] Integrar replay buffer al train loader.
- [ ] Evaluar Task-IL en tareas vistas.
- [ ] Guardar checkpoints y tablas de metricas.
- [ ] Calcular forgetting promedio.


## Paso 1: Dos Vistas por Imagen

Co2L necesita dos augmentations de la misma imagen para formar pares positivos en SupCon.

In [ ]:
# Reutiliza TwoViewWrapper desde dataloaders.py
task_id = 0
train_loader_task0 = seq_cifar.get_task_il_train_loader(task_id=task_id, use_buffer=False)
two_view_loader = torch.utils.data.DataLoader(
    TwoViewWrapper(train_loader_task0.dataset),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)
print(f"Two-view batches: {len(two_view_loader)}")

## Paso 2: Modelo con Projection Head

Separar classifier head (Task-IL) de projection head (loss contrastiva/distill).

In [ ]:
# TODO: mover estas clases a models.py
class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, dim=1)


class Co2LTaskILModel(nn.Module):
    def __init__(self, embedding_dim: int = 32, proj_dim: int = 128):
        super().__init__()
        self.backbone = CNN(in_channels=3, embedding_dim=embedding_dim)
        self.classifier = TaskIncrementalClassifier(self.backbone, embedding_dim=embedding_dim)
        self.projection_head = ProjectionHead(in_dim=embedding_dim, proj_dim=proj_dim)

    def forward_logits(self, x, task_id: int):
        return self.classifier(x, task_id=task_id)

    def forward_features(self, x):
        return self.backbone(x)

    def forward_projection(self, x):
        feat = self.forward_features(x)
        return self.projection_head(feat)


co2l_model = Co2LTaskILModel(embedding_dim=32, proj_dim=128).to(device)
print("Modelo Co2L inicializado.")

## Paso 3: Losses Co2L (Decoupled Representation-Classifier)

**Paper spec:** Co2L uses ONLY two losses (NO CE during representation learning):
- **L_sup_asym**: Asymmetric supervised contrastive loss (learn representations)
- **L_IRD**: Instance-wise relation distillation (preserve representations)

Classifier training happens AFTER representation learning ends (separate phase).

Formulas from paper (Method section):
- Total loss: $\mathcal{L} = \mathcal{L}^\text{sup}_\text{asym} + \lambda \cdot \mathcal{L}^\text{IRD}$
- IRD: $\mathcal{L}^\text{IRD} = \sum_{i=1}^{2N} -\mathbf{p}(\tilde{\mathbf{x}}_i; \psi^\text{past}, \kappa^*) \cdot \log \mathbf{p}(\tilde{\mathbf{x}}_i; \psi, \kappa)$


In [ ]:
# TODO: mover esta logica a losses.py
def supervised_contrastive_two_view_loss(z1, z2, labels, temperature=0.07):
    """SupCon simplificada con 2 vistas por muestra."""
    z = torch.cat([z1, z2], dim=0)
    y = torch.cat([labels, labels], dim=0)

    sim = torch.mm(z, z.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

    n = z.size(0)
    device_ = z.device
    eye = torch.eye(n, device=device_)

    pos_mask = (y.unsqueeze(0) == y.unsqueeze(1)).float() - eye
    exp_sim = torch.exp(sim) * (1.0 - eye)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True).clamp_min(1e-12))
    denom = pos_mask.sum(dim=1).clamp_min(1.0)
    loss = -((pos_mask * log_prob).sum(dim=1) / denom).mean()
    return loss


def instance_wise_relation_distillation_loss(student_z, teacher_z, kappa=0.07, kappa_star=0.07):
    """
    Instance-wise Relation Distillation (IRD) from Co2L paper.
    
    Matches normalized similarity relations between samples computed by student and teacher models.
    paper eq. (eqn:IRD): L_IRD = sum_i -p(x_i; psi_past, kappa*) · log p(x_i; psi, kappa)
    where p is the softmax similarity vector (excluding self-similarity).
    """
    B = student_z.size(0)
    device_ = student_z.device
    
    # Compute student similarity vectors
    student_sim = torch.mm(student_z, student_z.t()) / kappa  # [B, B]
    student_sim = student_sim - student_sim.max(dim=1, keepdim=True)[0].detach()
    
    # Compute teacher similarity vectors  
    teacher_sim = torch.mm(teacher_z, teacher_z.t()) / kappa_star  # [B, B]
    teacher_sim = teacher_sim - teacher_sim.max(dim=1, keepdim=True)[0].detach()
    
    # Mask out self-similarities
    eye = torch.eye(B, device=device_)
    denom_mask = 1.0 - eye
    
    # Student probabilities (softmax excluding self)
    student_exp = torch.exp(student_sim) * denom_mask
    student_prob = student_exp / student_exp.sum(dim=1, keepdim=True).clamp_min(1e-12)
    
    # Teacher probabilities (softmax excluding self)  
    teacher_exp = torch.exp(teacher_sim) * denom_mask
    teacher_prob = teacher_exp / teacher_exp.sum(dim=1, keepdim=True).clamp_min(1e-12)
    
    # IRD loss: cross-entropy between teacher and student probability vectors
    # L_IRD = sum_i -p_teacher · log(p_student)
    ird_loss = -(teacher_prob * torch.log(student_prob.clamp_min(1e-12))).sum(dim=1).mean()
    
    return ird_loss


@dataclass
class Co2LWeights:
    con: float = 1.0  # Contrastive learning weight
    dist: float = 1.0  # Distillation weight


def co2l_total_loss(z1, z2, teacher_z, labels, w: Co2LWeights, temperature=0.07, kappa=0.07, kappa_star=0.07):
    """
    Co2L total loss (NO CE loss - representation learning only).
    
    L = L_sup_asym + lambda * L_IRD
    
    Args:
        z1, z2: Student projections from two views
        teacher_z: Teacher projections (concatenated [t1, t2])
        labels: Labels for current batch
        w: Loss weights
        temperature: Temperature for supervised contrastive loss
        kappa, kappa_star: Temperatures for IRD (current and past models)
    """
    # Supervised contrastive loss (asymmetric - only current task positives)
    con_loss = supervised_contrastive_two_view_loss(z1, z2, labels, temperature=temperature)
    
    # Instance-wise relation distillation loss
    student_z_concat = torch.cat([z1, z2], dim=0)  # [2B, D]
    dist_loss = instance_wise_relation_distillation_loss(
        student_z_concat, teacher_z, kappa=kappa, kappa_star=kappa_star
    )
    
    # Total loss (NO CE)
    total = w.con * con_loss + w.dist * dist_loss
    return total, {"con": con_loss.item(), "dist": dist_loss.item()}


## Paso 4: Entrenamiento Fase 1 + Fase 2

**Fase 1:** Entrenar representación (backbone + projection head) con L_sup_asym + L_IRD (teacher congelado).
**Fase 2:** Entrenar classifier linealmente sobre representaciones congeladas con CE loss.


In [ ]:
# TODO: convertir a train_co2l.py cuando quede validado
def train_co2l_task(
    model,
    task_id,
    two_view_loader,
    device,
    epochs=10,
    lr=1e-3,
    weights=Co2LWeights(con=1.0, dist=1.0),
):
    """
    Train Co2L representation learning (decoupled from classifier).
    
    Co2L learns representations using L_sup_asym + L_IRD.
    NO classifier training during this phase.
    """
    model = model.to(device)
    
    # Create teacher snapshot BEFORE training
    teacher = deepcopy(model).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    # Train backbone + projection head only (NO classifier)
    model.backbone.train()
    model.projection_head.train()
    
    # Only optimize backbone and projection
    trainable_params = list(model.backbone.parameters()) + list(model.projection_head.parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=lr)

    history = []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        parts_sum = {"con": 0.0, "dist": 0.0}

        for (x1, x2), y in two_view_loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)

            # Forward pass (representation learning only)
            z1 = model.forward_projection(x1)
            z2 = model.forward_projection(x2)

            # Teacher forward (no grad)
            with torch.no_grad():
                t1 = teacher.forward_projection(x1)
                t2 = teacher.forward_projection(x2)
                teacher_z = torch.cat([t1, t2], dim=0)

            # Co2L loss (L_sup_asym + L_IRD, NO CE)
            loss, parts = co2l_total_loss(z1, z2, teacher_z, y, weights)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
            for k, v in parts.items():
                parts_sum[k] += v

        avg = running / len(two_view_loader)
        avg_parts = {k: v / len(two_view_loader) for k, v in parts_sum.items()}
        history.append({"epoch": epoch + 1, "loss": avg, **avg_parts})
        print(f"[Task {task_id}] epoch {epoch + 1}/{epochs} - loss={avg:.4f} con={avg_parts['con']:.4f} dist={avg_parts['dist']:.4f}")

    return model, history


In [ ]:
def train_co2l_classifier_phase2(
    model,
    task_id,
    train_loader,
    eval_loader,
    device,
    epochs=20,
    lr=0.1,
):
    """
    Phase 2: Train linear classifier on frozen representations.
    
    Backbone and projection head are frozen. Only classifier head is trained.
    Uses standard cross-entropy loss on frozen backbone features.
    
    Args:
        model: Co2LTaskILModel
        task_id: Task ID for this classifier head
        train_loader: Training DataLoader (single-view, not two-view)
        eval_loader: Evaluation DataLoader for validation
        device: torch device
        epochs: Number of training epochs
        lr: Learning rate for classifier
    """
    model = model.to(device)
    
    # Freeze backbone and projection head
    model.backbone.eval()
    model.projection_head.eval()
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    for p in model.projection_head.parameters():
        p.requires_grad_(False)
    
    # Ensure classifier has task head
    if not model.classifier.has_task(task_id):
        model.classifier.add_task(task_id=task_id, num_classes=2)
    
    # Only optimize classifier head
    classifier_params = [
        p for name, p in model.named_parameters() 
        if "classifier" in name and p.requires_grad
    ]
    optimizer = torch.optim.SGD(classifier_params, lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    history = []
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            
            # Frozen backbone features
            with torch.no_grad():
                features = model.forward_features(x)
            
            # Classifier forward (trainable)
            logits = model.classifier(features, task_id=task_id)
            loss = criterion(logits, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Evaluation phase
        model.eval()
        eval_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x, y in eval_loader:
                x, y = x.to(device), y.to(device)
                
                features = model.forward_features(x)
                logits = model.classifier(features, task_id=task_id)
                loss = criterion(logits, y)
                eval_loss += loss.item()
                
                _, preds = torch.max(logits, 1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        avg_eval_loss = eval_loss / len(eval_loader)
        eval_acc = correct / total if total > 0 else 0.0
        
        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "eval_loss": avg_eval_loss,
            "eval_acc": eval_acc,
        })
        
        if eval_acc > best_val_acc:
            best_val_acc = eval_acc
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"[Task {task_id}] Phase 2 epoch {epoch + 1}/{epochs} - "
                  f"train_loss={avg_train_loss:.4f} eval_loss={avg_eval_loss:.4f} eval_acc={eval_acc:.4f}")
    
    print(f"[Task {task_id}] Phase 2 complete. Best eval acc: {best_val_acc:.4f}")
    return model, history


## Co2L Training Summary

**Phase 1: Representation Learning** (frozen teacher, no classifier)
- Input: 2 augmented views per sample (2-view contrastive)
- Loss: $\mathcal{L}^\text{sup}_\text{asym} + \lambda \cdot \mathcal{L}^\text{IRD}$
- Trainable: backbone + projection_head
- Frozen: teacher snapshot from end of previous task

**Phase 2: Classifier Training** (frozen backbone, frozen projection)
- Input: single-view samples
- Loss: Standard cross-entropy on frozen backbone features
- Trainable: task-specific classifier head only
- Frozen: backbone + projection_head

**Key Insight:** Co2L decouples representation learning from classification.
- Phase 1 learns transferable representations via contrastive learning
- Phase 2 learns task-specific classifier on frozen representations
- This follows the paper's "without jointly trained classifier" principle


## Paso 5: Evaluacion y Forgetting

Evaluar Task-IL en todas las tareas vistas y computar forgetting promedio.

In [ ]:
def compute_average_forgetting(results_by_task: Dict[int, Dict[str, Dict[str, float]]]):
    """
    results_by_task[t]["task_k"]["acc"] = acc evaluada en k luego de entrenar t.
    """
    forgetting_values = []
    max_task = max(results_by_task.keys())

    for k in range(max_task):
        acc_history = []
        for t in range(k, max_task + 1):
            key = f"task_{k}"
            if key in results_by_task[t]:
                acc_history.append(results_by_task[t][key]["acc"])
        if len(acc_history) >= 2:
            forgetting_values.append(max(acc_history) - acc_history[-1])

    if len(forgetting_values) == 0:
        return 0.0
    return float(sum(forgetting_values) / len(forgetting_values))

## Paso 6: Bucle Completo 0->4

Entrenar secuencialmente y guardar checkpoint por tarea.

In [ ]:
# TODO: reemplazar warm-start con carga de pretrain real (task_0_pretrain)
# TODO: integrar replay real en dataloader de entrenamiento
all_histories = {}
all_results = {}

for task_id in range(NUM_TASKS):
    print(f"\n{'='*60}")
    print(f"=== Task {task_id} ===")
    print(f"{'='*60}")

    # ===== PHASE 1: Representation Learning =====
    print(f"\n[PHASE 1] Representation Learning (L_sup_asym + L_IRD)")
    
    # Two-view loader for Phase 1
    two_view_loader = seq_cifar.get_task_il_two_view_train_loader(
        task_id=task_id,
        use_buffer=False,
        num_workers=0,
    )

    # Train Phase 1: representations
    co2l_model, history_phase1 = train_co2l_task(
        model=co2l_model,
        task_id=task_id,
        two_view_loader=two_view_loader,
        device=device,
        epochs=5,
        lr=1e-3,
        weights=Co2LWeights(con=1.0, dist=1.0),
    )
    
    # ===== PHASE 2: Classifier Training =====
    print(f"\n[PHASE 2] Linear Classifier Training (frozen representations)")
    
    # Single-view loader for Phase 2 (standard, not two-view)
    train_loader_phase2 = seq_cifar.get_task_il_train_loader(
        task_id=task_id,
        use_buffer=False,
        num_workers=0,
    )
    
    # Get validation loader for Phase 2 monitoring
    val_loader = seq_cifar.get_task_il_eval_loader(task_id=task_id, num_workers=0)
    
    # Train Phase 2: classifier on frozen features
    co2l_model, history_phase2 = train_co2l_classifier_phase2(
        model=co2l_model,
        task_id=task_id,
        train_loader=train_loader_phase2,
        eval_loader=val_loader,
        device=device,
        epochs=20,
        lr=0.1,
    )
    
    # Store combined history
    all_histories[task_id] = {
        "phase1": history_phase1,
        "phase2": history_phase2,
    }

    # ===== EVALUATION =====
    print(f"\n[EVALUATION] Task-IL metrics")
    
    test_loaders = seq_cifar.get_task_il_test_loaders(task_id)
    results = evaluate_task_incremental(
        model=co2l_model.classifier,
        task_id=task_id,
        loaders=test_loaders,
        criterion=nn.CrossEntropyLoss(),
        device=device,
    )
    all_results[task_id] = results

    # Save checkpoint
    ckpt_path = f"{CO2L_CKPT_DIR}/task_{task_id}_model.pth"
    save_task_incremental_classifier(co2l_model.classifier, ckpt_path)
    print(f"✓ Checkpoint saved: {ckpt_path}")

avg_forgetting = compute_average_forgetting(all_results)
print(f"\n{'='*60}")
print(f"Final Average Forgetting: {avg_forgetting:.4f}")
print(f"{'='*60}")


## TODO Final de Refactor

- [ ] Pasar TwoViewWrapper a dataloaders.py con API limpia.
- [ ] Pasar ProjectionHead y Co2LTaskILModel a models.py.
- [ ] Pasar losses Co2L a losses.py.
- [ ] Crear train_co2l.py con train/eval utilitarios.
- [ ] Agregar guardado/carga de projection head en checkpoints.
- [ ] Agregar experimento de ablation (sin distill, sin replay, sin SupCon).
- [ ] Comparar Co2L vs LwF vs EWC en misma tabla.
- [ ] Documentar hiperparametros finales (tau, lambdas, buffer size).